In [1]:
import pandas as pd

# Load the dataset, skipping bad lines and using the Python engine for more robust parsing
df_meajor = pd.read_csv('meajor_cleaned_preprocessed (1).csv')

# Display the first 5 rows of the DataFrame
# display(df_meajor.head())

### 1. Data Preparation and Feature Engineering for NLP

To prepare the data for an NLP task, we'll combine the 'subject' and 'body' text, handle any missing values, and then use TF-IDF (Term Frequency-Inverse Document Frequency) to convert the text into a numerical format that machine learning models can understand.

In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine 'subject' and 'body' into a new 'text' column
# Fill NaN values with empty strings before combining to avoid errors
df_meajor['subject'] = df_meajor['subject'].fillna('')
df_meajor['body'] = df_meajor['body'].fillna('')
df_meajor['text'] = df_meajor['subject'] + ' ' + df_meajor['body']

# Display the first few rows with the new 'text' column
display(df_meajor[['subject', 'body', 'text', 'label']].head())

,subject,body,text,label
0,[ORGANIZATION] failover plan.,"Hi [NAME], \n\nTonight we are rolling out a n...","[ORGANIZATION] failover plan. Hi [NAME], \n\n...",0.0
1,RE: Intranet Site,"[NAME] r these new?\tIntranet Site\n\n[NAME],\...",RE: Intranet Site [NAME] r these new?\tIntrane...,0.0
2,FW: [ORGANIZATION] Company information,"[NAME]/[NAME],\n\nWe are currently trading und...",FW: [ORGANIZATION] Company information [NAME]/...,0.0
3,New Master Physical,[NAME] and [NAME] -\n\nAttached is a worksheet...,New Master Physical [NAME] and [NAME] -\n\nAtt...,0.0
4,FW: [ORGANIZATION]/Mirant GISB,FYI. Below is a copy of my communication with ...,FW: [ORGANIZATION]/Mirant GISB FYI. Below is a...,0.0


In [3]:
# Initialize TfidfVectorizer
# max_features can be adjusted to control the vocabulary size
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit and transform the 'text' column
X_tfidf = tfidf_vectorizer.fit_transform(df_meajor['text'])

# The target variable (labels)
y = df_meajor['label']

# Drop rows where 'y' contains NaN values and corresponding rows from X_tfidf
# First, identify the indices where y is not NaN
non_nan_indices = y.dropna().index

# Filter y to remove NaNs
y = y.loc[non_nan_indices]

# Filter X_tfidf to remove corresponding rows
X_tfidf = X_tfidf[non_nan_indices]

print(f"Shape of TF-IDF features (X_tfidf): {X_tfidf.shape}")
print(f"Shape of target labels (y): {y.shape}")

Shape of TF-IDF features (X_tfidf): (108684, 5000)
Shape of target labels (y): (108684,)


In [4]:
# Re-run the modified cell to ensure 'y' and 'X_tfidf' are cleaned
# This cell was modified previously to drop NaN values from 'y' and 'X_tfidf'
# It is executed here to reflect those changes.
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(df_meajor['text'])
y = df_meajor['label']

non_nan_indices = y.dropna().index
y = y.loc[non_nan_indices]
X_tfidf = X_tfidf[non_nan_indices]

print(f"Shape of TF-IDF features (X_tfidf) after NaN removal: {X_tfidf.shape}")
print(f"Shape of target labels (y) after NaN removal: {y.shape}")

Shape of TF-IDF features (X_tfidf) after NaN removal: (108684, 5000)
Shape of target labels (y) after NaN removal: (108684,)


### 2. Model Training and Evaluation

Now, we'll split the TF-IDF features and labels into training and testing sets. We will then train an XGBoost Classifier on the training data and evaluate its performance on the test data.

In [5]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (86947, 5000)
X_test shape: (21737, 5000)
y_train shape: (86947,)
y_test shape: (21737,)


In [6]:
import time
import xgboost as xgb
# Assuming X_train, y_train are already defined from previous cells (e.g., f284510f)

# Initialize XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# Train the model
print("Training XGBoost model...")
start_time_xgb = time.time()
xgb_model.fit(X_train, y_train)
end_time_xgb = time.time()
xgb_training_time = end_time_xgb - start_time_xgb
print("XGBoost model training complete.")

print(f"XGBoost training time: {xgb_training_time:.2f} seconds")

Training XGBoost model...


c:\Users\bmey2\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:02:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost model training complete.
XGBoost training time: 60.56 seconds


In [7]:
# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Display a more detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9677
Precision: 0.9647
Recall: 0.9621
F1-Score: 0.9634

Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.97      0.97     12130
         1.0       0.96      0.96      0.96      9607

    accuracy                           0.97     21737
   macro avg       0.97      0.97      0.97     21737
weighted avg       0.97      0.97      0.97     21737



In [8]:
# The evaluation metrics for XGBoost should already be available from cell d7cb99f8
# Re-executing it to ensure variables are in scope if needed.

# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Evaluate the model
accuracy_xgb = accuracy_score(y_test, y_pred)
precision_xgb = precision_score(y_test, y_pred)
recall_xgb = recall_score(y_test, y_pred)
f1_xgb = f1_score(y_test, y_pred)

print(f"XGBoost Accuracy: {accuracy_xgb:.4f}")
print(f"XGBoost Precision: {precision_xgb:.4f}")
print(f"XGBoost Recall: {recall_xgb:.4f}")
print(f"XGBoost F1-Score: {f1_xgb:.4f}")

XGBoost Accuracy: 0.9677
XGBoost Precision: 0.9647
XGBoost Recall: 0.9621
XGBoost F1-Score: 0.9634


### 3. Model Comparison: Random Forest and MLP

Let's compare the performance of our XGBoost model with a Random Forest Classifier and a Multi-layer Perceptron (MLP) Classifier using the same TF-IDF features.

#### Random Forest Classifier

In [9]:
from sklearn.ensemble import RandomForestClassifier
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
# Assuming X_train, y_train, X_test, y_test are already defined from previous cells (e.g., f284510f)

# Initialize Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Train the model
print("Training Random Forest Classifier...")
start_time_rf = time.time()
rf_model.fit(X_train, y_train)
end_time_rf = time.time()
rf_training_time = end_time_rf - start_time_rf
print("Random Forest Classifier training complete.")

print(f"Random Forest training time: {rf_training_time:.2f} seconds")

# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate the model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"\nRandom Forest Accuracy: {accuracy_rf:.4f}")
print(f"Random Forest Precision: {precision_rf:.4f}")
print(f"Random Forest Recall: {recall_rf:.4f}")
print(f"Random Forest F1-Score: {f1_rf:.4f}")

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Training Random Forest Classifier...
Random Forest Classifier training complete.
Random Forest training time: 56.67 seconds

Random Forest Accuracy: 0.9793
Random Forest Precision: 0.9847
Random Forest Recall: 0.9683
Random Forest F1-Score: 0.9764

Random Forest Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.98     12130
         1.0       0.98      0.97      0.98      9607

    accuracy                           0.98     21737
   macro avg       0.98      0.98      0.98     21737
weighted avg       0.98      0.98      0.98     21737



#### Multi-layer Perceptron (MLP) Classifier

In [10]:
from sklearn.neural_network import MLPClassifier
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
# Assuming X_train, y_train, X_test, y_test are already defined from previous cells (e.g., f284510f)

# Initialize MLP Classifier
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=200, random_state=42, verbose=True)

# Train the model
print("\nTraining MLP Classifier...")
start_time_mlp = time.time()
mlp_model.fit(X_train, y_train)
end_time_mlp = time.time()
mlp_training_time = end_time_mlp - start_time_mlp
print("MLP Classifier training complete.")

print(f"MLP training time: {mlp_training_time:.2f} seconds")

# Make predictions on the test set
y_pred_mlp = mlp_model.predict(X_test)

# Evaluate the model
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
precision_mlp = precision_score(y_test, y_pred_mlp)
recall_mlp = recall_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp)

print(f"\nMLP Accuracy: {accuracy_mlp:.4f}")
print(f"\nMLP Precision: {precision_mlp:.4f}")
print(f"\nMLP Recall: {recall_mlp:.4f}")
print(f"\nMLP F1-Score: {f1_mlp:.4f}")

print("\nMLP Classification Report:")
print(classification_report(y_test, y_pred_mlp))


Training MLP Classifier...


Iteration 1, loss = 0.17975344
Iteration 2, loss = 0.07508770
Iteration 3, loss = 0.06316265
Iteration 4, loss = 0.05726380
Iteration 5, loss = 0.05335771
Iteration 6, loss = 0.05070744
Iteration 7, loss = 0.04862587
Iteration 8, loss = 0.04638354
Iteration 9, loss = 0.04441512
Iteration 10, loss = 0.04245555
Iteration 11, loss = 0.03992006
Iteration 12, loss = 0.03760048
Iteration 13, loss = 0.03484767
Iteration 14, loss = 0.03173325
Iteration 15, loss = 0.02872564
Iteration 16, loss = 0.02545926
Iteration 17, loss = 0.02210159
Iteration 18, loss = 0.01910592
Iteration 19, loss = 0.01665533
Iteration 20, loss = 0.01421898
Iteration 21, loss = 0.01235711
Iteration 22, loss = 0.01064861
Iteration 23, loss = 0.00962885
Iteration 24, loss = 0.00878751
Iteration 25, loss = 0.00781665
Iteration 26, loss = 0.00730476
Iteration 27, loss = 0.00691665
Iteration 28, loss = 0.00670344
Iteration 29, loss = 0.00652816
Iteration 30, loss = 0.00649335
Iteration 31, loss = 0.00610278
Iteration 32, lo

### POSSIBLE EXTENSION Considerations for Convolutional Neural Network (CNN)

To effectively use a Convolutional Neural Network (CNN) for this task, the input data format typically needs to be different. Instead of TF-IDF vectors, CNNs for NLP usually take sequences of word embeddings as input. This would involve:

1.  **Tokenization**: Breaking down text into individual words or subword units.
2.  **Word Embeddings**: Converting these tokens into dense vector representations (e.g., Word2Vec, GloVe, FastText, or contextual embeddings like BERT/DistilBERT).
3.  **Padding/Truncation**: Ensuring all input sequences have the same length for batch processing.

This approach would require a significant change to our feature engineering step. If you'd like to explore this, please let me know, and we can start by preparing the data for a CNN.

### 4. Data Preparation for Convolutional Neural Network (CNN)

To prepare the text data for a CNN, we need to transform it into sequences of integers, where each integer represents a word. This typically involves tokenization and padding.

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Define parameters for tokenization and padding
VOCAB_SIZE = 10000  # Max number of words to keep, based on word frequency
MAX_SEQUENCE_LENGTH = 200 # Max length of each sequence, shortens or pads longer/shorter reviews

# Align df_meajor['text'] with the filtered y by using non_nan_indices
# This ensures X_cnn and y have the same number of samples
text_for_cnn = df_meajor['text'].loc[non_nan_indices]

# Initialize Tokenizer
# Filters out punctuation, leaves alphanumeric characters and spaces
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<unk>') # <unk> for out-of-vocabulary words

# Fit tokenizer on the 'text' data
tokenizer.fit_on_texts(text_for_cnn)

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(text_for_cnn)

# Pad sequences to ensure uniform length
X_cnn = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"Shape of CNN input data (X_cnn): {X_cnn.shape}")
print(f"Shape of target labels (y): {y.shape}")

# Display a sample of tokenized and padded sequences
print("\nSample of padded sequences (first 2 entries):")
print(X_cnn[:2])

Shape of CNN input data (X_cnn): (108684, 200)
Shape of target labels (y): (108684,)

Sample of padded sequences (first 2 entries):
[[   8    1  536  211    9  772   35   32 4957   70    6   81  238  549
    73   10    4    9   31  317    3   25  529 1031   25  223    2  533
   900 3039   25   42  352    8  423  238   97   10 2323   25   25   42
   429    2  130   21 1140   84   45  107  900   82   16    6 1276   36
     2 1559   19  517 2300 2217   25   42  257   10  687 7419  532    2
    68   19  517   96    3    8 2893   10   48  125   96   25   33    6
    38   88   48   27 4841    3    8  201   60   90   40   82   16  348
   497  146  546  317   94    9    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
    

### 5. Build and Train the CNN Model

Now we will define a Convolutional Neural Network (CNN) architecture using Keras, train it on our preprocessed text data, and evaluate its performance.

In [12]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Split the CNN-prepared data into training and testing sets
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(X_cnn, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train_cnn shape: {X_train_cnn.shape}")
print(f"X_test_cnn shape: {X_test_cnn.shape}")
print(f"y_train_cnn shape: {y_train_cnn.shape}")
print(f"y_test_cnn shape: {y_test_cnn.shape}")

X_train_cnn shape: (86947, 200)
X_test_cnn shape: (21737, 200)
y_train_cnn shape: (86947,)
y_test_cnn shape: (21737,)


In [13]:
# Model parameters
EMBEDDING_DIM = 100 # Dimension of the word embeddings

# Build the CNN model
cnn_model = Sequential([
    # Embedding layer: converts word indices to dense vectors
    # input_dim: size of the vocabulary (VOCAB_SIZE)
    # output_dim: dimension of the dense embedding (EMBEDDING_DIM)
    # input_length: length of input sequences (MAX_SEQUENCE_LENGTH)
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),

    # 1D Convolutional layer: extracts local features from the sequences
    # filters: number of convolution filters (output dimensionality)
    # kernel_size: length of the 1D convolution window
    # activation: activation function
    Conv1D(filters=128, kernel_size=5, activation='relu'),

    # Global Max Pooling layer: downsamples the input by taking the maximum value over time steps
    # This helps in creating a fixed-size output regardless of input length after convolution
    GlobalMaxPooling1D(),

    # Dense layer: standard fully connected neural network layer
    Dense(64, activation='relu'),

    # Dropout layer: helps prevent overfitting by randomly setting a fraction of input units to 0 at each update during training
    Dropout(0.5),

    # Output layer: for binary classification, a single neuron with sigmoid activation
    Dense(1, activation='sigmoid')
])

# Compile the model
# optimizer: Adam is a popular choice for deep learning models
# loss: binary_crossentropy is suitable for binary classification
# metrics: accuracy to monitor performance during training
cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
cnn_model.summary()

c:\Users\bmey2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
import time
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Load the dataset (redundant but for robustness if kernel resets)
df_meajor = pd.read_csv('meajor_cleaned_preprocessed (1).csv')

# Combine 'subject' and 'body' into a new 'text' column
df_meajor['subject'] = df_meajor['subject'].fillna('')
df_meajor['body'] = df_meajor['body'].fillna('')
df_meajor['text'] = df_meajor['subject'] + ' ' + df_meajor['body']

# The target variable (labels)
y_cnn_raw = df_meajor['label']

# Drop rows where 'y_cnn_raw' contains NaN values and filter text_for_cnn accordingly
non_nan_indices = y_cnn_raw.dropna().index
y = y_cnn_raw.loc[non_nan_indices] # Use 'y' for consistency with other models
text_for_cnn = df_meajor['text'].loc[non_nan_indices]

# Define parameters for tokenization and padding
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 100

# Initialize Tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<unk>')
tokenizer.fit_on_texts(text_for_cnn)
sequences = tokenizer.texts_to_sequences(text_for_cnn)
X_cnn = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

# Split the CNN-prepared data into training and testing sets
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(X_cnn, y, test_size=0.2, random_state=42, stratify=y)

# Build the CNN model
cnn_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Train the CNN model
print("\nTraining CNN model...")
start_time_cnn = time.time()
history = cnn_model.fit(
    X_train_cnn, y_train_cnn, epochs=5, batch_size=32, validation_split=0.1, verbose=1
)
end_time_cnn = time.time()
cnn_training_time = end_time_cnn - start_time_cnn
print("CNN model training complete.")

print(f"CNN training time: {cnn_training_time:.2f} seconds")


Training CNN model...
Epoch 1/5


c:\Users\bmey2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


2446/2446 ━━━━━━━━━━━━━━━━━━━━ 52s 21ms/step - accuracy: 0.9590 - loss: 0.1077 - val_accuracy: 0.9810 - val_loss: 0.0603
Epoch 2/5
2446/2446 ━━━━━━━━━━━━━━━━━━━━ 56s 23ms/step - accuracy: 0.9897 - loss: 0.0321 - val_accuracy: 0.9826 - val_loss: 0.0543
Epoch 3/5
2446/2446 ━━━━━━━━━━━━━━━━━━━━ 52s 21ms/step - accuracy: 0.9951 - loss: 0.0152 - val_accuracy: 0.9832 - val_loss: 0.0634
Epoch 4/5
2446/2446 ━━━━━━━━━━━━━━━━━━━━ 53s 22ms/step - accuracy: 0.9975 - loss: 0.0090 - val_accuracy: 0.9815 - val_loss: 0.0959
Epoch 5/5
2446/2446 ━━━━━━━━━━━━━━━━━━━━ 53s 22ms/step - accuracy: 0.9979 - loss: 0.0066 - val_accuracy: 0.9821 - val_loss: 0.1036
CNN model training complete.
CNN training time: 266.47 seconds


In [15]:
print("\n--- Model Training Time Summary ---")
print(f"XGBoost Training Time: {xgb_training_time:.2f} seconds")
print(f"Random Forest Training Time: {rf_training_time:.2f} seconds")
print(f"MLP Training Time: {mlp_training_time:.2f} seconds")
print(f"CNN Training Time: {cnn_training_time:.2f} seconds")


--- Model Training Time Summary ---
XGBoost Training Time: 60.56 seconds
Random Forest Training Time: 56.67 seconds
MLP Training Time: 742.67 seconds
CNN Training Time: 266.47 seconds


In [16]:
# Make predictions on the test set
y_pred_cnn_proba = cnn_model.predict(X_test_cnn)
y_pred_cnn = (y_pred_cnn_proba > 0.5).astype(int)

# Evaluate the model
accuracy_cnn = accuracy_score(y_test_cnn, y_pred_cnn)
precision_cnn = precision_score(y_test_cnn, y_pred_cnn)
recall_cnn = recall_score(y_test_cnn, y_pred_cnn)
f1_cnn = f1_score(y_test_cnn, y_pred_cnn)

print(f"\nCNN Accuracy: {accuracy_cnn:.4f}")
print(f"CNN Precision: {precision_cnn:.4f}")
print(f"CNN Recall: {recall_cnn:.4f}")
print(f"CNN F1-Score: {f1_cnn:.4f}")

print("\nCNN Classification Report:")
print(classification_report(y_test_cnn, y_pred_cnn))

680/680 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step

CNN Accuracy: 0.9820
CNN Precision: 0.9812
CNN Recall: 0.9779
CNN F1-Score: 0.9796

CNN Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.98     12130
         1.0       0.98      0.98      0.98      9607

    accuracy                           0.98     21737
   macro avg       0.98      0.98      0.98     21737
weighted avg       0.98      0.98      0.98     21737



In [17]:
# The evaluation metrics for CNN should already be available from cell f74bac09
# Re-executing it to ensure variables are in scope if needed.

# Make predictions on the test set
y_pred_cnn_proba = cnn_model.predict(X_test_cnn)
y_pred_cnn = (y_pred_cnn_proba > 0.5).astype(int)

# Evaluate the model
accuracy_cnn = accuracy_score(y_test_cnn, y_pred_cnn)
precision_cnn = precision_score(y_test_cnn, y_pred_cnn)
recall_cnn = recall_score(y_test_cnn, y_pred_cnn)
f1_cnn = f1_score(y_test_cnn, y_pred_cnn)

print(f"CNN Accuracy: {accuracy_cnn:.4f}")
print(f"CNN Precision: {precision_cnn:.4f}")
print(f"CNN Recall: {recall_cnn:.4f}")
print(f"CNN F1-Score: {f1_cnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
CNN Accuracy: 0.9820
CNN Precision: 0.9812
CNN Recall: 0.9779
CNN F1-Score: 0.9796


In [18]:
import pandas as pd

# Collect all scores in a dictionary
performance_data = {
    'Model': ['XGBoost', 'Random Forest', 'MLP', 'CNN'],
    'Accuracy': [accuracy_xgb, accuracy_rf, accuracy_mlp, accuracy_cnn],
    'Precision': [precision_xgb, precision_rf, precision_mlp, precision_cnn],
    'Recall': [recall_xgb, recall_rf, recall_mlp, recall_cnn],
    'F1-Score': [f1_xgb, f1_rf, f1_mlp, f1_cnn],
    'Training Time (seconds)': [xgb_training_time, rf_training_time, mlp_training_time, cnn_training_time]
}

# Create a DataFrame for better display
performance_df = pd.DataFrame(performance_data)

print("\n--- Model Performance and Training Time Comparison ---")
display(performance_df.set_index('Model').round(4))


--- Model Performance and Training Time Comparison ---


,Accuracy,Precision,Recall,F1-Score,Training Time (seconds)
Model,,,,,
XGBoost,0.9677,0.9647,0.9621,0.9634,60.5569
Random Forest,0.9793,0.9847,0.9683,0.9764,56.6750
MLP,0.9751,0.9734,0.9702,0.9718,742.6664
CNN,0.9820,0.9812,0.9779,0.9796,266.4746


The Neural Netwroks Deep Learning took 100 times longer than RF for comparable scores. To get the near perfect precision from the deep learning from the reading a LSTM and GRU will have to be added, which will take additional time. Both are still subject to fail at zero day phishing emails but the NLPs are faster to adjust.

Code obtained from scikit machine learning documentation (XGB, RF, MLP) and tensorflow for the Neural Network Steps. Help improving the code and printing checks for Google's Gemini